# 8. Models del llenguatge amb n-grames 

Tractament de Dades Textuals i Codificades — Eixample Clínic, 2026

Pol Pastells, ppastells@eixampleclinic.es

## Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from nltk import ngrams

np.random.seed(42)

plt.rcParams["figure.dpi"] = 100
%matplotlib inline
%config InlineBackend.figure_format='retina'

In [ ]:
df = pd.read_csv("noms_cat.csv", keep_default_na=False)
llista_noms = df["Nom"]
noms = [f"#{nom}#" for nom in llista_noms]
freqs = df.freq.to_numpy()

In [ ]:
caracters = ["#"] + sorted(list(set("".join(df.Nom))))
ncar = len(caracters)
c2i = {c: i for i, c in enumerate(caracters)}
i2c = {i: c for i, c in enumerate(caracters)}

## Model amb recompte de bigrames

In [ ]:
N2 = np.zeros((ncar, ncar), dtype=int)
for i, nom in enumerate(noms):
    for c1, c2 in ngrams(nom, 2):
        ix1 = c2i[c1]
        ix2 = c2i[c2]
        N2[ix1, ix2] += freqs[i]

In [ ]:
P2 = N2 / N2.sum(axis=1, keepdims=True)

In [ ]:
for _ in range(10):
    ix = 0
    nom = ""
    while True:
        ix = np.argmax(np.random.multinomial(1, P2[ix]))
        if ix == 0:
            break
        nom += i2c[ix]
    print(nom)

## Trigrames

In [ ]:
N3 = np.zeros((ncar, ncar, ncar), dtype=int)
for i, nom in enumerate(noms):
    for c1, c2, c3 in ngrams(nom, 3):
        N3[c2i[c1], c2i[c2], c2i[c3]] += freqs[i]

In [ ]:
N3_sum = N3.sum(axis=2, keepdims=True)
P3 = np.divide(N3, N3_sum, out=np.zeros(N3.shape, dtype=float), where=N3_sum != 0)

In [ ]:
for _ in range(10):
    # Primer caràcter a partir de bigrames
    ix1 = 0
    ix2 = np.argmax(np.random.multinomial(1, P2[ix1]))
    nom = i2c[ix2]

    # Resta amb trigrames
    while True:
        ix3 = np.argmax(np.random.multinomial(1, P3[ix1, ix2]))
        if ix3 == 0:
            break
        nom += i2c[ix3]
        ix1 = ix2
        ix2 = ix3

    print(nom)

## Exercici A - Continuació amb noms de persona

### Exercici A1: adapta el codi de generació de noms amb trigrames perquè generi noms d'home o de dona

Crea una funció `genera_noms` que rebi un sol argument `sexe` i retorni un únic nom

Deixarem les matrius de probabilitat com a variables globals.

Et caldran dues matrius per homes (PH2, PH3) i dues per dones (PD2, PD3).


### Solució

In [ ]:
# columna nom# per facilitar-nos la vida
df["nom#"] = "#" + df.Nom + "#"

In [ ]:
noms_home = df[df.Sexe == "H"]["nom#"].to_list()
noms_dona = df[df.Sexe == "D"]["nom#"].to_list()

freqs_home = df[df.Sexe == "H"]["freq"].to_list()
freqs_dona = df[df.Sexe == "D"]["freq"].to_list()

In [ ]:
def get_P2(noms, freqs, c2i):
    ncar = len(c2i)
    N2 = np.zeros((ncar, ncar), dtype=int)
    for i, nom in enumerate(noms):
        for c1, c2 in zip(nom, nom[1:]):
            N2[c2i[c1], c2i[c2]] += freqs[i]

    N2_sum = N2.sum(axis=1, keepdims=True)
    P2 = np.divide(N2, N2_sum, out=np.zeros(N2.shape, dtype=float), where=N2_sum != 0)
    return P2

In [ ]:
def get_P3(noms, freqs, c2i):
    ncar = len(c2i)
    N3 = np.zeros((ncar, ncar, ncar), dtype=int)
    for i, nom in enumerate(noms):
        for c1, c2, c3 in ngrams(nom, 3):
            N3[c2i[c1], c2i[c2], c2i[c3]] += freqs[i]

    N3_sum = N3.sum(axis=2, keepdims=True)
    P3 = np.divide(N3, N3_sum, out=np.zeros(N3.shape, dtype=float), where=N3_sum != 0)
    return P3

In [ ]:
# Deixem les matrius de probabilitat com a variables globals
PH2 = get_P2(noms_home, freqs_home, c2i)
PH3 = get_P3(noms_home, freqs_home, c2i)
PD2 = get_P2(noms_dona, freqs_dona, c2i)
PD3 = get_P3(noms_dona, freqs_dona, c2i)

In [ ]:
def genera_noms(sexe):
    """
    Genera noms de dona (D) o home (H)
    """
    if sexe == "H":
        P2 = PH2
        P3 = PH3
    elif sexe == "D":
        P2 = PD2
        P3 = PD3
    else:
        raise ValueError("sexe ha de ser H o D")

    # Primer caràcter a partir de bigrames
    ix1 = 0
    ix2 = np.argmax(np.random.multinomial(1, P2[ix1]))
    nom = i2c[ix2]

    # Resta amb trigrames
    while True:
        ix3 = np.argmax(np.random.multinomial(1, P3[ix1, ix2]))
        if ix3 == 0:
            break
        nom += i2c[ix3]
        ix1 = ix2
        ix2 = ix3

    return nom

In [ ]:
for _ in range(10):
    print(genera_noms("H"))

In [ ]:
for _ in range(10):
    print(genera_noms("D"))

### Exercici A2: amplia la funció `genera_noms` perquè faci servir 4-grames

### Solució

In [ ]:
def get_P4(noms, freqs, c2i):
    ncar = len(c2i)
    N4 = np.zeros((ncar,) * 4, dtype=int)
    for i, nom in enumerate(noms):
        for c1, c2, c3, c4 in ngrams(nom, 4):
            N4[c2i[c1], c2i[c2], c2i[c3], c2i[c4]] += freqs[i]

    N4_sum = N4.sum(axis=3, keepdims=True)
    P4 = np.divide(N4, N4_sum, out=np.zeros(N4.shape, dtype=float), where=N4_sum != 0)
    return P4

In [ ]:
PH4 = get_P4(noms_home, freqs_home, c2i)
PD4 = get_P4(noms_dona, freqs_dona, c2i)

In [ ]:
def genera_noms(sexe):
    if sexe == "H":
        P2 = PH2
        P3 = PH3
        P4 = PH4
    elif sexe == "D":
        P2 = PD2
        P3 = PD3
        P4 = PD4
    else:
        raise ValueError("sexe ha de ser H o D")

    # Primer caràcter a partir de bigrames
    ix1 = 0
    ix2 = np.argmax(np.random.multinomial(1, P2[ix1]))
    nom = i2c[ix2]

    # Segon caràcter a partir de trigrames
    ix3 = np.argmax(np.random.multinomial(1, P3[ix1, ix2]))
    if ix3 == 0:
        return nom
    else:
        nom += i2c[ix3]

    # Resta amb 4-grames
    while True:
        ix4 = np.argmax(np.random.multinomial(1, P4[ix1, ix2, ix3]))
        if ix4 == 0:
            return nom
        nom += i2c[ix4]
        ix1 = ix2
        ix2 = ix3
        ix3 = ix4

Tot i ser només un cas, existeix el nom _E_, d'un sol caràcter. 

Per tant, la probabilitat que `ix3` sigui 0 no és nula. Per aixo tenim el primer `return nom`

In [ ]:
PD3[c2i["#"], c2i["e"], c2i["#"]]

In [ ]:
for _ in range(10):
    print(genera_noms("H"))

In [ ]:
for _ in range(10):
    print(genera_noms("D"))

#### Resultat? 

A mi em semblen millors!

### *Exercici A3: amplia la funció `genera_noms` perquè faci servir n-grames

Com que generar les matrius de probabilitat pot ser un procés lent, busca com es fa servir el paquet `tqdm` per mostrar una barra amb el progrés.

Fes el codi en general, i prova'l amb 5-grames.

1. Fes primer una funció `get_Pn` que retorni les matrius de probabilitat corresponents a n-grames, partint de `get_P4`.
2. Genera i guarda les matrius de P2 a P5 corresponents a homes i dones en un vector per cada sexe (PnsH, PnsD).
3. Crea la funció `genera_noms(Pns, n)`, que rebi com a input Pns i el nombre d'ngrames que farà servir per la generació. 
 

### Pistes

1. Per `genera_noms`, pots fer servir `deque` per guardar els indexs ix1, ix2... en un array que automàticament mantingui només `n-1` índexs.

In [ ]:
from collections import deque

ixs = deque([0], maxlen=3)
ixs.append(1)
print(ixs)
ixs.append(2)
print(ixs)
ixs.append(15)
print(ixs)
ixs.append(3)
print(ixs)
ixs.append(14)
print(ixs)

2. Pots indexar un array de numpy amb una tuple, si ho fas directament amb deque (igual que amb una llista o un array, ho tractarà com a índexs diferents)
```python
PH4[tuple(ixs)].shape
```

3. Fixa't en què es repeteix a `genera_noms` de l'exercici 2. Agrupa-ho dins un `while True` amb un únic `return nom`

### Solució

In [ ]:
from tqdm.auto import tqdm

In [ ]:
def get_Pn(noms, freqs, c2i, n):
    """Retorna la matriu Pn corresponent a n-grames"""
    ncar = len(c2i)
    N = np.zeros((ncar,) * n, dtype=int)
    for i, nom in enumerate(noms):
        for cs in ngrams(nom, n):
            ixs = tuple(c2i[c] for c in cs)
            N[ixs] += freqs[i]

    N_sum = N.sum(axis=n - 1, keepdims=True)
    P = np.divide(N, N_sum, out=np.zeros(N.shape, dtype=float), where=N_sum != 0)
    return P

In [ ]:
def generar_Pns(noms, freqs, c2i, n):
    """crea una llista amb les matrius (tensors) Pn
    n: N de l'N-grama màxim
    """
    Pns = []
    for i in tqdm(range(2, n + 1)):
        Pns.append(get_Pn(noms, freqs, c2i, i))
    return Pns

In [ ]:
# fem un exemple amb 5-grames
PnsH = generar_Pns(noms_home, freqs_home, c2i, 5)
PnsD = generar_Pns(noms_dona, freqs_dona, c2i, 5)

In [ ]:
from collections import deque


def genera_noms(Pns, n):
    assert n <= len(Pns) + 1
    ixs = deque([0], maxlen=n - 1)
    nom = ""

    i = 0
    while True:
        ix = np.argmax(np.random.multinomial(1, Pns[i][tuple(ixs)]))
        if ix == 0:
            return nom
        else:
            ixs.append(ix)
            nom += i2c[ix]

        # Augmentem l'ordre del model fins arribar a n-grames
        if i + 2 < n:
            i += 1

In [ ]:
for _ in range(10):
    print(genera_noms(PnsH, n=5))

In [ ]:
for _ in range(10):
    print(genera_noms(PnsD, n=5))

## Exercici B - Principis actius

Ara proveu a canviar les dades a una llista de principis actius. Ens podem baixar un excel amb medicacions de l'AEMPS: https://listadomedicamentos.aemps.gob.es/Medicamentos.xls

In [ ]:
# pot ser que us calgui instalar aquesta llibreria per poder llegir excel
# ! pip install openpyxl

In [ ]:
medicaments = pd.read_excel("Medicamentos.xls")

In [ ]:
# cada medicament té una columna amb una llista dels principis actius
medicaments["Principios Activos"]

In [ ]:
# podem passar-ho a llista
medicaments["Principios Activos"].str.split(", ").head()

In [ ]:
# i aplanar-ho
from itertools import chain

principis = list(chain.from_iterable(medicaments["Principios Activos"].str.split(", ").to_list()))
len(principis), principis[:20]

In [ ]:
from collections import Counter

principis_count = Counter(principis)
len(principis_count), principis_count.most_common(10)

In [ ]:
llista_noms = list(principis_count.keys())
llista_noms = [nom.lower() for nom in llista_noms]

noms = [f"#{nom}#" for nom in llista_noms]
freqs = np.array(list(principis_count.values()))

caracters = ["#"] + sorted(list(set("".join(llista_noms))))
ncars = len(caracters)
c2i = {c: i for i, c in enumerate(caracters)}
i2c = {i: c for i, c in enumerate(caracters)}

In [ ]:
# Quants principis únics tenim?
print(f"Total entrades: {len(principis)}")
print(f"Principis únics: {len(principis_count)}")
print(f"Caràcters únics: {ncars}")

# Quins caràcters nous apareixen vs els noms catalans?
print(caracters)

### Solució amb 5-grames

In [ ]:
Pns_principis = generar_Pns(noms, freqs, c2i, 5)

In [ ]:
for _ in range(10):
    print(genera_noms(Pns_principis, n=5))